# 02 - 用 PINN 求解一维 Poisson 方程

目标方程：

$$u''(x) = -\pi^2 \sin(\pi x), \quad x \in [0,1]$$

边界条件：

$$u(0)=0, \quad u(1)=0$$

解析解：

$$u(x)=\sin(\pi x)$$

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=1, out_dim=1, hidden_dim=48, depth=4):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers += [nn.Linear(hidden_dim, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

## 二阶导数残差

Poisson 方程需要 `u''(x)`，所以要连续调用两次 `torch.autograd.grad`。

In [ ]:
def poisson_residual(model, x):
    x.requires_grad_(True)
    u = model(x)
    du_dx = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    d2u_dx2 = torch.autograd.grad(du_dx, x, torch.ones_like(du_dx), create_graph=True)[0]
    forcing = -(math.pi**2) * torch.sin(math.pi * x)
    return d2u_dx2 - forcing

def exact_solution(x):
    return torch.sin(math.pi * x)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

x_b = torch.tensor([[0.0], [1.0]], device=device)
u_b = torch.zeros_like(x_b)

for epoch in range(7000):
    x_f = torch.rand(256, 1, device=device)
    r = poisson_residual(model, x_f)
    loss_phys = torch.mean(r**2)
    loss_bc = torch.mean((model(x_b) - u_b)**2)
    loss = loss_phys + 50.0 * loss_bc

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    history.append([loss.item(), loss_phys.item(), loss_bc.item()])
    if epoch % 1000 == 0:
        print(f"epoch={epoch:4d}, loss={loss.item():.3e}, phys={loss_phys.item():.3e}, bc={loss_bc.item():.3e}")

In [ ]:
x = torch.linspace(0, 1, 400, device=device).reshape(-1, 1)
with torch.no_grad():
    u_pred = model(x)
    u_exact = exact_solution(x)

history_t = torch.tensor(history)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(x.cpu(), u_exact.cpu(), label="exact")
plt.plot(x.cpu(), u_pred.cpu(), "--", label="PINN")
plt.legend()
plt.title("Solution")

plt.subplot(1, 3, 2)
plt.plot(x.cpu(), torch.abs(u_pred - u_exact).cpu())
plt.title("Absolute error")

plt.subplot(1, 3, 3)
plt.semilogy(history_t[:, 0], label="total")
plt.semilogy(history_t[:, 1], label="physics")
plt.semilogy(history_t[:, 2], label="boundary")
plt.legend()
plt.title("Loss terms")
plt.tight_layout()
plt.show()

print("max abs error =", torch.max(torch.abs(u_pred - u_exact)).item())

## 练习

1. 把边界损失权重 `50.0` 改小，观察边界条件是否变差。
2. 把采样点数量 `256` 改成 `32`，观察残差和误差。
3. 尝试把解析解改成 `sin(2*pi*x)`，重新推导右端 forcing。